In [2]:
from ultralytics import YOLO

# 1. Load YOUR trained model (Point this to your best weights file!)
model = YOLO("../results/runs/run_04_yolo26s/weights/best.pt")

# 2. Run inference on your Validation images
# save=True tells YOLO to draw the boxes and save the pictures
# conf=0.25 tells it to only draw blocks it is at least 25% sure about
print("Running AI predictions...")
results = model.predict(
    source="../data/ml_dataset/val/images", 
    save=True, 
    conf=0.25, 
    imgsz=256,
    project="../results/runs",
    name="val_predictions"
)
print("Done! Go check the output folder.")

Running AI predictions...

image 1/390 /Users/jjwill18/Desktop/europa-chaos-blocks/notebooks/../data/ml_dataset/val/images/Chaos_E_0_768_flipLR.jpg: 256x256 13 chaos_blocks, 79.1ms
image 2/390 /Users/jjwill18/Desktop/europa-chaos-blocks/notebooks/../data/ml_dataset/val/images/Chaos_E_0_768_flipUD.jpg: 256x256 16 chaos_blocks, 79.4ms
image 3/390 /Users/jjwill18/Desktop/europa-chaos-blocks/notebooks/../data/ml_dataset/val/images/Chaos_E_0_768_orig.jpg: 256x256 9 chaos_blocks, 74.8ms
image 4/390 /Users/jjwill18/Desktop/europa-chaos-blocks/notebooks/../data/ml_dataset/val/images/Chaos_E_0_768_rot180.jpg: 256x256 14 chaos_blocks, 75.0ms
image 5/390 /Users/jjwill18/Desktop/europa-chaos-blocks/notebooks/../data/ml_dataset/val/images/Chaos_E_0_768_rot270.jpg: 256x256 5 chaos_blocks, 77.3ms
image 6/390 /Users/jjwill18/Desktop/europa-chaos-blocks/notebooks/../data/ml_dataset/val/images/Chaos_E_0_768_rot90.jpg: 256x256 5 chaos_blocks, 75.2ms
image 7/390 /Users/jjwill18/Desktop/europa-chaos-blocks

In [9]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from sahi.predict import get_sliced_prediction
from sahi import AutoDetectionModel
import rasterio


# 1. Paths (Update these!)
model_path = "../results/runs/run_04_yolo26s/weights/best.pt"
raw_image_path = "../data/preprocessed/images/Chaos_H.tif"
output_dir = "../results/runs/macro_predictions"
os.makedirs(output_dir, exist_ok=True)

# 2. Load your YOLO model into SAHI
detection_model = AutoDetectionModel.from_pretrained(
    model_type='yolo26',
    model_path=model_path,
    confidence_threshold=0.30,
    device="mps"
)

print(f"Loading {os.path.basename(raw_image_path)}...")

# 3. Read and safely normalize the scientific TIFF
with rasterio.open(raw_image_path) as src:
    img_data = src.read(1)
    
    # Normalize the Float32 scientific data to 8-bit visual data (0-255)
    img_data = np.nan_to_num(img_data, nan=np.nanmin(img_data))
    vmin, vmax = np.percentile(img_data, [2, 98])
    if vmax > vmin:
        img_data = np.clip((img_data - vmin) / (vmax - vmin), 0, 1)
    
    # Convert to standard 8-bit image array
    img_8bit = (img_data * 255).astype(np.uint8)
    
    # Convert to RGB (OpenCV expects 3 color channels to draw colored boxes)
    img_rgb = cv2.cvtColor(img_8bit, cv2.COLOR_GRAY2RGB)

print("Running sliced macro prediction... (This might take a minute)")

# 4. Run Sliced Inference
# Change the slice size to match your training chips exactly!
result = get_sliced_prediction(
    img_rgb,
    detection_model,
    slice_height=256,       # <--- Match your training size!
    slice_width=256,        # <--- Match your training size!
    overlap_height_ratio=0.2,
    overlap_width_ratio=0.2
)

print("Drawing predictions...")

# 5. Manually draw the bounding boxes
output_img = img_rgb.copy()
for prediction in result.object_prediction_list:
    # Get bounding box coordinates
    bbox = prediction.bbox
    x1, y1, x2, y2 = int(bbox.minx), int(bbox.miny), int(bbox.maxx), int(bbox.maxy)
    
    # Draw a green rectangle
    cv2.rectangle(output_img, (x1, y1), (x2, y2), (0, 255, 0), thickness=2)

# 6. Save the final image using Matplotlib
output_file = os.path.join(output_dir, f"macro_pred_{os.path.basename(raw_image_path)}.jpg")

plt.figure(figsize=(15, 15))
plt.imshow(output_img)
plt.axis('off')
plt.title(f"Macro Prediction: {len(result.object_prediction_list)} Blocks Found")
plt.savefig(output_file, bbox_inches='tight', dpi=300)
plt.close()

print(f"Done! Saved to: {output_file}")


Loading Chaos_H.tif...
Running sliced macro prediction... (This might take a minute)
Performing prediction on 16 slices.
Drawing predictions...
Done! Saved to: ../results/runs/macro_predictions/macro_pred_Chaos_H.tif.jpg
